In [1]:
# Gemma 4 native video evaluation
# Model: google/gemma-4-12B-it

from google.colab import userdata
from huggingface_hub import login
from pathlib import Path

import json
import time
import sys
import torch

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig
)

In [2]:
## check if cuda is available and GPU

print("CUDA available:", torch.cuda.is_available())
print("Using GPU:", torch.cuda.get_device_name(0))

CUDA available: True
Using GPU: Tesla T4


In [3]:
## authenticate with huggingface

hf_token = userdata.get("hf_token")
login(token=hf_token)

In [4]:
%pip install -q -U bitsandbytes accelerate

In [5]:
# Load Gemma 4 12B using 4-bit quantization

from transformers import BitsAndBytesConfig

model_id = "google/gemma-4-12B-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForMultimodalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Gemma 4 loaded successfully!")

processor_config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.09k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 23.9GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

Gemma 4 loaded successfully!


In [6]:
# Numerical overflow in Gemma 4 vision processing

import torch

vision = model.model.embed_vision

# Use Float32 for vision normalization
vision.patch_ln1.float()
vision.patch_ln2.float()
vision.pos_norm.float()

# Use Float32 computation for quantized vision projections
vision.patch_dense.compute_dtype = torch.float32

vision.multimodal_embedder.embedding_projection.compute_dtype = torch.float32

print("Float32 vision fix applied successfully!")
print("Vision normalization:", vision.patch_ln1.weight.dtype)
print("First projection:", vision.patch_dense.compute_dtype)
print(
    "Second projection:",
    vision.multimodal_embedder.embedding_projection.compute_dtype
)

Float32 vision fix applied successfully!
Vision normalization: torch.float32
First projection: torch.float32
Second projection: torch.float32


In [7]:
from pathlib import Path

repo_folder = Path("/content/Physics_LLM_Test")

# Download the repository
if not repo_folder.exists():
    !git clone https://github.com/ShRuwali/Physics_LLM_Test.git /content/Physics_LLM_Test

video_folder = repo_folder / "Problems" / "Video"

output_folder = repo_folder / "Result" / "open_weight_models" / "video" / "gemma_4"
metadata_folder = output_folder / "metadata"

output_folder.mkdir(parents=True, exist_ok=True)
metadata_folder.mkdir(parents=True, exist_ok=True)

file_names = sorted(video_folder.glob("*.mp4"))

print("Total videos found:", len(file_names))
print("Output folder:", output_folder)
print("Metadata folder:", metadata_folder)

Cloning into '/content/Physics_LLM_Test'...
remote: Enumerating objects: 856, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 856 (delta 0), reused 0 (delta 0), pack-reused 854 (from 1)
Receiving objects: 100% (856/856), 8.01 MiB | 5.79 MiB/s, done.
Resolving deltas: 100% (414/414), done.
Total videos found: 24
Output folder: /content/Physics_LLM_Test/Result/open_weight_models/video/gemma_4
Metadata folder: /content/Physics_LLM_Test/Result/open_weight_models/video/gemma_4/metadata


In [8]:
import sys

# Locate video prompts
sys.path.insert(0, str(video_folder))

from video_prompts import regular_system_prompt, detail_system_prompt

# Use both prompt dictionaries
prompt_dict = [regular_system_prompt, detail_system_prompt]

print("Regular prompts:", len(regular_system_prompt))
print("Detailed prompts:", len(detail_system_prompt))
print("Total expected evaluations:",
      len(regular_system_prompt) + len(detail_system_prompt))

{'Problem_1_Regular_Q': 'You are a physics expert. From the video provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the video.', 'Problem_1_Obvious_Q': 'You are a physics expert. From the video provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the video.', 'Problem_1_non_Obvious_Q': 'You are a physics expert. From the video provide a brief description and calculate the total time taken by the block to travel 5 meters. All the necessary values required are given in the video.', 'Problem_2_Regular_Q': 'You are a physics expert. From the video provide a brief description and calculate the time period of oscillation for small displacements. All the necessary values required are given in the video.', 'Problem_2_Obvious_Q': 'You are a physics expert. From the video provide a brief description and calculate th

In [9]:
%%time

import json
import time
import torch
import transformers

# Model and generation settings
model_name = "gemma-4"
max_new_tokens = 1024
completed_evaluations = 0

# Verify that every research prompt has a matching video
video_keys = {file_path.stem.lower() for file_path in file_names}

for prompt_dictionary in [regular_system_prompt, detail_system_prompt]:
    missing_videos = [
        key for key in prompt_dictionary
        if key.lower() not in video_keys
    ]

    if missing_videos:
        raise ValueError(
            f"Missing video files for prompts: {missing_videos}"
        )

expected_evaluations = (
    len(regular_system_prompt) + len(detail_system_prompt)
)

if expected_evaluations != 40:
    raise ValueError(
        f"Expected 40 evaluations, found {expected_evaluations}"
    )

# Evaluate all videos using regular and detailed prompts
for prompt_dictionary, prompt_type in [
    (regular_system_prompt, "regular_system_prompt"),
    (detail_system_prompt, "detail_system_prompt")
]:

    for file_path in file_names:

        file_key = file_path.stem

        # Match video filenames with prompt dictionary keys
        # Ignore differences in capitalization
        matching_key = next(
            (
                key for key in prompt_dictionary
                if key.lower() == file_key.lower()
            ),
            None
        )

        # Detailed prompts are only available for selected videos
        if matching_key is None:
            continue

        system_prompt = prompt_dictionary[matching_key]

        # Prepare the original MP4 video and research prompt
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "video", "path": str(file_path)},
                    {"type": "text", "text": system_prompt}
                ]
            }
        ]

        # Process video using Gemma 4's native video processor
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=False,
            return_dict=True,
            return_tensors="pt"
        ).to(model.device)

        input_length = inputs["input_ids"].shape[-1]

        # Record the actual number of processed video frames
        actual_video_frames = int(
            inputs["num_frames_per_video"].sum().item()
        )

        # Generate response and measure generation time
        torch.cuda.synchronize()
        start_time = time.perf_counter()

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False
            )

        torch.cuda.synchronize()
        total_time_seconds = time.perf_counter() - start_time

        # Decode only the generated response tokens
        generated_tokens = outputs[0, input_length:]
        output_tokens = generated_tokens.numel()

        response = processor.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        if not response:
            raise ValueError(
                f"Empty response generated for {file_key} ({prompt_type})"
            )

        # Save the model response as Markdown
        output_filename = (
            f"{model_name}_{file_key}_{prompt_type}.md"
        )

        output_file_path = output_folder / output_filename

        output_file_path.write_text(
            response,
            encoding="utf-8"
        )

        # Record experimental metadata
        metadata = {
            "requested_model": model_id,
            "problem_question": file_key,
            "prompt_name": system_prompt,
            "total_output_tokens": int(output_tokens),
            "total_time_taken": float(total_time_seconds),
            "video_file": file_path.name,
            "video_input_method": "native_video",
            "video_processor": type(processor.video_processor).__name__,
            "processor_num_frames": processor.video_processor.num_frames,
            "actual_video_frames": actual_video_frames,
            "transformers_version": transformers.__version__,
            "generation_max_new_tokens": max_new_tokens,
            "generation_do_sample": False,
            "quantization": "4-bit NF4",
            "model_dtype": "float16",
            "bnb_4bit_compute_dtype": "float16",
            "bnb_4bit_double_quant": True,
            "vision_precision_workaround": "Float32"
        }

        # Save metadata as JSON
        metadata_filename = (
            f"{model_name}_{file_key}_{prompt_type}.json"
        )

        metadata_file_path = metadata_folder / metadata_filename

        metadata_file_path.write_text(
            json.dumps(
                metadata,
                indent=4,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )

        completed_evaluations += 1

        # Print completion status
        print(
            f"Completed:::{model_name}_{file_key}_{prompt_type}",
            flush=True
        )

# Display final evaluation summary
print(f"\nTotal completed evaluations: {completed_evaluations}")

if completed_evaluations == expected_evaluations:
    print("All 40 native video evaluations completed successfully!")
else:
    print("Warning: Some evaluations are missing.")


Completed:::gemma-4_Problem_1_Obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_1_Regular_Q_regular_system_prompt
Completed:::gemma-4_Problem_1_non_obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_2_Obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_2_Regular_Q_regular_system_prompt
Completed:::gemma-4_Problem_2_non_obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_3_Obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_3_Regular_Q_regular_system_prompt
Completed:::gemma-4_Problem_3_non_obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_4_Obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_4_Regular_Q_regular_system_prompt
Completed:::gemma-4_Problem_4_non_obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_5_Obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_5_Regular_Q_regular_system_prompt
Completed:::gemma-4_Problem_5_non_obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_6_Obvious_Q_regular_sy